# Modelforge OpenMM Wrapper

## Simple Example

This notebook demonstrates the steps required to use a potential trained using modelforge as a `PythonFroce` within OpenMM.

### Load modelforge potential file
First let us import some various modules from modelforge we require for reading in a checkpoint file from a trained model:

In [1]:
from modelforge.potential.potential import load_inference_model_from_checkpoint

# helper functions to load in the example model checkpoint file included in the repository
from modelforge.utils.io import get_path_string
from modelforge.ase.tests import data

# checkpoint file is saved in tests/data
checkpoint_file_path = get_path_string(data) + "/model.ckpt"
potential = load_inference_model_from_checkpoint(checkpoint_file_path, jit=False)


2026-06-10 09:36:53.837 | DEBUG    | modelforge.potential.potential:generate_potential:889 - training_parameter=None
2026-06-10 09:36:53.838 | DEBUG    | modelforge.potential.potential:generate_potential:890 - potential_parameter=SchNetParameters(potential_name='SchNet', only_unique_pairs=False, core_parameter=CoreParameter(number_of_radial_basis_functions=128, maximum_interaction_radius=0.49999999999999994, number_of_interaction_modules=9, number_of_filters=256, shared_interactions=True, activation_function_parameter=ActivationFunctionConfig(activation_function_name='ShiftedSoftplus', activation_function_arguments=None, activation_function=ShiftedSoftplus()), featurization=Featurization(properties_to_featurize=['atomic_number', 'per_system_total_charge'], atomic_number=AtomicNumber(maximum_atomic_number=101, number_of_per_atom_features=512), atomic_period=AtomicPeriod(maximum_period=8, number_of_per_period_features=32), atomic_group=AtomicGroup(maximum_group=18, number_of_per_group_fe

### Generate OpenMM topology
Next, we will create an OpenMM topology object.  This uses a simple helper function that generates a molecule using RDKit. 

In [2]:
from modelforge.openmm.examples.helper_functions import openmm_topology_from_smiles

topology, positions = openmm_topology_from_smiles(smiles='NCCCCCO', optimize=True)
atomic_numbers = [atom.element.atomic_number for atom in topology.atoms()]


### Define OpenMM PythonForce

To allow us to interface with OpenMM, we wrap the modelforge potential, using the `generate_compute` function, and then pass this to OpenMM's PythonForce. 

In [3]:
from modelforge.openmm.potential import generate_compute
from openmm import PythonForce

comp = generate_compute(potential=potential, atomic_numbers=atomic_numbers)
system_force = PythonForce(comp)

### OpenMM simulation setup

We then need to define a standard simulation workflow for OpenMM.

In [4]:
import openmm
from openmm.unit import (
    kelvin,
    picosecond,
    femtosecond,
    nanometer,
    kilojoules_per_mole,
)
from openmm.app import PDBFile

# define the systme
system = openmm.System()
for atom in topology.atoms():
    system.addParticle(atom.element.mass)

# add the system_force instance defined above that wraps modelforge potential for PythonFroce
system.addForce(system_force)


import sys
from openmm import LangevinMiddleIntegrator
from openmm.app import Simulation, StateDataReporter, DCDReporter

# Create an integrator with a time step of 1 fs
temperature = 298.15 * kelvin
frictionCoeff = 1.0 / femtosecond
timeStep = 1.0 * femtosecond
integrator = LangevinMiddleIntegrator(temperature, frictionCoeff, timeStep)

# Create a simulation and set the initial positions and velocities
simulation = Simulation(topology, system, integrator)
simulation.context.setPositions(positions)

reporter = StateDataReporter(
    file=sys.stdout,
    reportInterval=100,
    step=True,
    time=True,
    potentialEnergy=True,
    temperature=True,
)
simulation.reporters.append(reporter)

# write out a trajectory and pdb file so we can visualize
simulation.reporters.append(DCDReporter('trajectory.dcd', 100))

state = simulation.context.getState(getPositions=True, getEnergy=True)
print("initial_energy ", state.getPotentialEnergy())


with open('initial_config.pdb', 'w') as f:
    PDBFile.writeFile(simulation.topology, state.getPositions(), f)
simulation.step(10000)

initial_energy  -7316.2392578125 kJ/mol
#"Step","Time (ps)","Potential Energy (kJ/mole)","Temperature (K)"
100,0.10000000000000007,-7636.90869140625,316.8905372438258
200,0.20000000000000015,-7635.92919921875,283.0822283193324
300,0.3000000000000002,-7713.6083984375,372.161755262278
400,0.4000000000000003,-7768.1728515625,301.5721881708677
500,0.5000000000000003,-7797.27880859375,342.2783498511418
600,0.6000000000000004,-7839.87109375,343.66904177790155
700,0.7000000000000005,-7825.86962890625,213.45527501315138
800,0.8000000000000006,-7833.30712890625,303.61009766575904
900,0.9000000000000007,-7829.3115234375,253.44578106722705
1000,1.0000000000000007,-7829.138671875,270.58924697810613
1100,1.0999999999999897,-7858.927734375,292.46296794151556
1200,1.1999999999999786,-7897.3427734375,246.47158355482748
1300,1.2999999999999676,-7877.3310546875,335.54615542943486
1400,1.3999999999999566,-7918.61962890625,280.87095260628564
1500,1.4999999999999456,-7876.724609375,469.8241334775827
1600,1